In [ ]:
# Se cargan 60 días observados de una ciudad ficticia antes del corte de información.

import numpy as np
import pandas as pd

observed = pd.read_csv("../data/observed_active_cases.csv")
observed.tail()

In [ ]:
# Se implementa un SIR con fallecimientos que conserva la población de la ciudad.

population = 100_000
recovery_rate = 0.08
death_rate = 0.001

def simulate_sir(infection_rate, initial_infected, forecast_days):
    susceptible, infected, recovered, deceased = population - initial_infected, initial_infected, 0.0, 0.0
    records = []
    for day in range(forecast_days):
        records.append({"day": day, "susceptible": susceptible, "infected": infected, "recovered": recovered, "deceased": deceased})
        new_infections = infection_rate * infected * susceptible / population
        new_recoveries = recovery_rate * infected
        new_deaths = death_rate * infected
        susceptible -= new_infections
        infected += new_infections - new_recoveries - new_deaths
        recovered += new_recoveries
        deceased += new_deaths
    return pd.DataFrame(records)

In [ ]:
# ¿Qué tasa constante reproduce mejor los casos activos disponibles hasta hoy?

initial_infected = observed.loc[0, "active_cases"]
fit_comparison = []
for infection_rate in np.round(np.arange(0.10, 0.41, 0.01), 2):
    simulation = simulate_sir(infection_rate, initial_infected, len(observed))
    fit_comparison.append({"infection_rate": infection_rate, "observed_mse": ((simulation["infected"] - observed["active_cases"]) ** 2).mean()})
fit_comparison = pd.DataFrame(fit_comparison)
baseline_rate = fit_comparison.loc[fit_comparison["observed_mse"].idxmin(), "infection_rate"]
baseline_rate

In [ ]:
# Se proyectan escenarios de transmisión sin seleccionar una política óptima.

scenarios = {"transmision_actual": baseline_rate, "transmision_moderada": baseline_rate * 0.75, "transmision_alta": baseline_rate * 1.15}
forecasts = []
for scenario, infection_rate in scenarios.items():
    simulation = simulate_sir(infection_rate, initial_infected, 180)
    simulation["scenario"] = scenario
    simulation["infection_rate"] = infection_rate
    simulation["required_beds"] = simulation["infected"] * 0.05
    forecasts.append(simulation)
forecasts = pd.concat(forecasts, ignore_index=True)
forecasts.head()

In [ ]:
# ¿En qué día ocurre el pico y cuántas camas se requieren en cada escenario?

bed_capacity = 1_500
scenario_peaks = forecasts.loc[forecasts.groupby("scenario")["infected"].idxmax()].copy()
scenario_peaks = scenario_peaks[["scenario", "infection_rate", "day", "infected", "required_beds"]].rename(columns={"day": "peak_day", "infected": "peak_active_cases", "required_beds": "peak_required_beds"})
scenario_peaks["bed_capacity"] = bed_capacity
scenario_peaks["bed_gap"] = scenario_peaks["peak_required_beds"] - bed_capacity
scenario_peaks

In [ ]:
# Se conservan pronósticos, picos y supuestos para verificar el escenario construido.

import json
from pathlib import Path

submission_dir = Path("../submission")
forecasts.to_csv(submission_dir / "forecasts.csv", index=False)
scenario_peaks.to_csv(submission_dir / "scenario_peaks.csv", index=False)
fit_comparison.to_csv(submission_dir / "fit_comparison.csv", index=False)
with (submission_dir / "model_assumptions.json").open("w", encoding="utf-8") as file:
    json.dump({"population": population, "observation_cutoff_day": int(observed["day"].max()), "forecast_days": 180, "recovery_rate": recovery_rate, "death_rate": death_rate, "hospitalization_rate": 0.05, "bed_capacity": bed_capacity, "baseline_infection_rate": baseline_rate}, file, indent=2)